# 🏷️ Gemini Labeler — 新聞情感標注器

**角色：Teacher / Labeler**

使用 Gemini 2.5 Flash API 對原始新聞資料進行 **1–10 分情感標注**，產出供 Qwen 使用的訓練集。

---

| 項目 | 說明 |
|------|------|
| **輸入** | `raw_news.jsonl`（欄位：`stock`, `input`）|
| **輸出** | `gemini_labeled.jsonl`（新增欄位：`gemini_label`）|
| **模型** | `gemini-2.5-flash` |
| **評分** | 1（重大利空）→ 10（極度利多）|

```
raw_news.jsonl
      ↓  [本 Notebook]
gemini_labeled.jsonl
      ↓  [下一步：qwen_predictor_colab.ipynb]
qwen_predictions.jsonl
```

## 📦 Step 1 — 安裝相依套件

In [29]:
!pip install google-generativeai -q
print('✅ 套件安裝完成')

✅ 套件安裝完成


## ⚙️ Step 2 — 設定區

In [1]:
import os

API_KEY = os.getenv('GEMINI_API_KEY', 'YOUR_GEMINI_API_KEY')
#API_KEY = os.getenv('GEMINI_API_KEY', 'YOUR_GEMINI_API_KEY')

INPUT_FILE  = 'data/jsonl/gemini_label/prelabeled.jsonl'
OUTPUT_FILE = 'data/jsonl/gemini_label/finish_labeled.jsonl'
RESUME_FILE = 'gemini_labeled_resume.txt'

# MODEL_NAME = 'gemini-2.5-flash-lite'
MODEL_NAME = 'gemini-3.1-flash-lite'

REQUESTS_PER_MINUTE = 14
SLEEP_BETWEEN_CALLS = 60 / REQUESTS_PER_MINUTE 

MAX_RETRIES      = 3
RETRY_BASE_DELAY = 10  # 秒，指數退避基數（10s → 20s → 40s）

GEN_CONFIG = {
    "max_output_tokens": 120,
    "temperature": 0.1,
    "top_p": 0.9,
}

print(f'✅ 設定完成')
print(f'   模型   : {MODEL_NAME}')
print(f'   輸入   : {INPUT_FILE}')
print(f'   輸出   : {OUTPUT_FILE}')
print(f'   速率   : {REQUESTS_PER_MINUTE} RPM（間隔 {SLEEP_BETWEEN_CALLS:.1f}s）')

✅ 設定完成
   模型   : gemini-3.1-flash-lite
   輸入   : data/jsonl/gemini_label/prelabeled.jsonl
   輸出   : data/jsonl/gemini_label/finish_labeled.jsonl
   速率   : 14 RPM（間隔 4.3s）


## 📝 Step 3 — Logger 初始化

In [2]:
import logging

# 同時輸出到 console 與 log 檔
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler('gemini_labeler.log', encoding='utf-8'),
    ],
)
log = logging.getLogger(__name__)
log.info('Logger 初始化完成')

2026-07-03 01:20:38,057 [INFO] Logger 初始化完成


## 🧠 Step 4 — System Prompt 定義

與 Qwen Predictor 使用**完全相同**的評分標準，確保兩者對齊。

In [3]:
import json
from pathlib import Path

# 1. 定義原始的 System Prompt 基礎結構
BASE_SYSTEM_PROMPT = """
你是一位台灣股市量化分析師。
請分析以下新聞對「指定股票」的股價情緒影響，給出 1 到 10 的整數情緒分數。

【判斷維度（請綜合考量）】
- 影響程度：消息對基本面（營收、獲利、訂單、產能、競爭力）的衝擊大小
- 確定性：是「已發生/已證實」還是「預期/傳聞/可能」
- 影響範圍：僅該公司、整個產業、還是大盤層級
- 時間框架：短期（1個月內反映）還是長期（需數季才顯現）

【評分標準與範例】

 1 分：重大利空，股價可能大跌
   例：財務造假、重大違約/跳票、巨額虧損且超乎預期、下市/重整風險、
       董座或高層涉及重大刑事案件、工廠重大火災或停產且復原時間長、
       訴訟敗訴需鉅額賠償。

 2 分：明顯利空
   例：單季獲利或營收明顯不如市場預期、法說會大幅下修財測、
       重大客戶/訂單流失或延遲、信評或外資大幅調降評等與目標價、
       裁員或關廠等成本結構惡化訊號。

 3 分：輕微利空
   例：營收月減或年減但在合理範圍內、外資小幅調降目標價、
       原物料成本上升但公司表示可部分轉嫁、產業面出現溫和的負面雜音。

 4 分：偏空中性，預期有小幅負面壓力
   例：消息本身偏負面但對基本面影響極小或屬一次性費用、
       同業出現利空但對該公司影響有限、技術面或籌碼面的輕微負面訊息。

 5 分：完全中性，無明確影響
   例：例行性公告（股東會召開、董監事補選、一般人事異動）、
       與股價無直接關聯的企業形象/公益新聞、單純背景報導不含新資訊。

 6 分：偏多中性，長期可能正向但短期難反映
   例：長期利多但短期難反映在財報上（如簽署MOU、初期合作意向）、
       產業趨勢面偏正向但該公司受惠程度尚不明確、小幅度的正面雜音。

 7 分：輕微利多，已有可觀察到的正面訊號
   例：營收月增或年增且符合或略優於預期、取得新客戶或小型新訂單、
       外資小幅調升目標價、毛利率小幅改善。

 8 分：明顯利多
   例：單季獲利或營收明顯優於市場預期、法說會上修財測、
       取得指標性大客戶訂單、外資大幅調升評等或目標價、新產品/新產能開出。

 9 分：強力利多，股價可能大漲
   例：取得遠超預期的鉅額訂單或長約、重大技術突破並開始量產、
       公司大幅上修財測且幅度顯著、獲利創歷史新高且超乎市場預期甚多。

10 分：極度利多，歷史性重大正面事件
   例：被收購且收購價大幅溢價、與重量級企業達成排他性策略合作、
       公司營運模式出現結構性轉變且確定性極高、其他足以改變公司基本面定位的重大事件。

【特殊情況處理】
- 若新聞內容與該股票無直接或間接關聯，給 5 分，並將 confidence 設為1。
- 若新聞為「傳聞」、「市場預期」、「分析師臆測」而非已證實事實，
  分數不應達到極端值（1、2、9、10），且 confidence 應相對降低。
- 若新聞同時包含利多與利空訊息，請綜合評估「淨影響方向與強度」後給分。

【信心度判斷原則】
- 0.9~1.0：新聞內容明確、數據具體、已發生事實、對公司影響清楚
- 0.7~0.85：新聞明確但影響程度需解讀、或屬中大型事件
- 0.5~0.7：新聞較模糊、需推論、或混合訊息
- 0.3~0.5：傳聞、分析師意見、市場臆測、與公司關聯間接
- 0.3以下：與指定股票幾乎無關、或新聞極度模糊

【輸出格式】（嚴格遵守，僅輸出 JSON，不加任何說明文字）
{
  "sentiment_score": <1 到 10 的整數>,
  "confidence": <0.0 到 1.0 的浮點數，代表你對此判斷的信心>
}
""".strip()

# 2. 讀取錯題本並轉換為 Few-shot 提示詞
ERROR_BOOK_FILE = 'data/jsonl/gemini_label/error_book_gemini.json'
few_shot_context = ""

error_book_path = Path(ERROR_BOOK_FILE)
if error_book_path.exists():
    try:
        with open(error_book_path, 'r', encoding='utf-8') as f:
            content = f.read().strip()
            if content:
                # 支援標準 JSON 陣列 (Array) 或 JSONL 格式的錯題本
                cases = json.loads(content) if content.startswith('[') else [json.loads(line) for line in content.splitlines() if line.strip()]
                
                valid_cases = []
                for idx, case in enumerate(cases, start=1):
                    news_input = case.get('input', '').strip()
                    correct_score = case.get('correct_score')
                    reason = case.get('reason', '未提供理由').strip()
                    
                    # 僅保留基本的新聞內容存在檢查
                    if news_input:
                        valid_cases.append(
                            f"【修正案例 {idx}】\n"
                            f"新聞內容：{news_input}\n"
                            f"正確分數：{int(correct_score)} 分\n"
                            f"修正原因：{reason}\n"
                            f"引導輸出：{{\"sentiment_score\": {int(correct_score)}, \"confidence\": 1.0}}\n"
                            f"──────────────────────────────────────────────────"
                        )
                
                if valid_cases:
                    few_shot_context = (
                        "\n\n==================================================\n"
                        "【量化標注修正歷史案例 (Few-shot)】\n"
                        "以下是你過去情感標注發生偏誤，經由量化分析師校正後的黃金標準案例。\n"
                        "請細讀這些「修正原因」與「正確分數」之對應邏輯，並在本次預測中，遇到相似情境時比照此標準給予客觀的分數：\n\n"
                        + "\n\n".join(valid_cases)
                    )
                    print(f"🎯 成功載入 {len(valid_cases)} 筆修正完畢的錯題作為 Few-shot 提示詞！")
    except Exception as e:
        print(f"⚠️ 讀取錯題本時發生錯誤: {e}，本次標注將使用原始 Zero-shot 進行。")

# 3. 合併為最終的 SYSTEM_PROMPT
SYSTEM_PROMPT = BASE_SYSTEM_PROMPT + few_shot_context

print('✅ System Prompt 設定完成')
print('─' * 50)
print(SYSTEM_PROMPT)

✅ System Prompt 設定完成
──────────────────────────────────────────────────
你是一位台灣股市量化分析師。
請分析以下新聞對「指定股票」的股價情緒影響，給出 1 到 10 的整數情緒分數。

【判斷維度（請綜合考量）】
- 影響程度：消息對基本面（營收、獲利、訂單、產能、競爭力）的衝擊大小
- 確定性：是「已發生/已證實」還是「預期/傳聞/可能」
- 影響範圍：僅該公司、整個產業、還是大盤層級
- 時間框架：短期（1個月內反映）還是長期（需數季才顯現）

【評分標準與範例】

 1 分：重大利空，股價可能大跌
   例：財務造假、重大違約/跳票、巨額虧損且超乎預期、下市/重整風險、
       董座或高層涉及重大刑事案件、工廠重大火災或停產且復原時間長、
       訴訟敗訴需鉅額賠償。

 2 分：明顯利空
   例：單季獲利或營收明顯不如市場預期、法說會大幅下修財測、
       重大客戶/訂單流失或延遲、信評或外資大幅調降評等與目標價、
       裁員或關廠等成本結構惡化訊號。

 3 分：輕微利空
   例：營收月減或年減但在合理範圍內、外資小幅調降目標價、
       原物料成本上升但公司表示可部分轉嫁、產業面出現溫和的負面雜音。

 4 分：偏空中性，預期有小幅負面壓力
   例：消息本身偏負面但對基本面影響極小或屬一次性費用、
       同業出現利空但對該公司影響有限、技術面或籌碼面的輕微負面訊息。

 5 分：完全中性，無明確影響
   例：例行性公告（股東會召開、董監事補選、一般人事異動）、
       與股價無直接關聯的企業形象/公益新聞、單純背景報導不含新資訊。

 6 分：偏多中性，長期可能正向但短期難反映
   例：長期利多但短期難反映在財報上（如簽署MOU、初期合作意向）、
       產業趨勢面偏正向但該公司受惠程度尚不明確、小幅度的正面雜音。

 7 分：輕微利多，已有可觀察到的正面訊號
   例：營收月增或年增且符合或略優於預期、取得新客戶或小型新訂單、
       外資小幅調升目標價、毛利率小幅改善。

 8 分：明顯利多
   例：單季獲利或營收明顯優於市場預期、法說會上修財測、
       取得指標性大客戶訂單、外資大幅調升評等或目標價、

## 🔧 Step 5 — 輔助函式定義

包含：
- `init_model()` — 初始化 Gemini 模型
- `call_gemini_with_retry()` — 帶指數退避的 API 呼叫
- `load_resume_line()` / `save_resume_line()` — 斷點續跑支援

In [4]:
import json
import time
from pathlib import Path
from datetime import datetime
import google.generativeai as genai


def init_model() -> genai.GenerativeModel:
    """初始化 Gemini 模型，開啟 JSON 輸出模式"""
    genai.configure(api_key=API_KEY)
    model = genai.GenerativeModel(
        MODEL_NAME,
        generation_config={
            'response_mime_type': 'application/json',
            'temperature': 0.1,       # 標注任務需穩定，調低溫度
            'max_output_tokens': 128,
        },
    )
    log.info(f'模型初始化完成：{MODEL_NAME}')
    return model


def call_gemini_with_retry(model: genai.GenerativeModel, prompt: str) -> dict:
    """呼叫 Gemini API，失敗時以指數退避重試（最多 MAX_RETRIES 次）"""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = model.generate_content(
                prompt,
                generation_config={
                    "response_mime_type": "application/json",
                    "max_output_tokens": 120,
                    "temperature": 0.1,
                }
            )

            # ✅ 安全檢查（避免 candidates 為空）
            if not response.candidates:
                raise ValueError("沒有回傳 candidates")

            candidate = response.candidates[0]

            # 🔴 關鍵：檢查是否被截斷
            if candidate.finish_reason == "MAX_TOKENS":
                raise ValueError("輸出被截斷（MAX_TOKENS）")

            if not candidate.content.parts:
                raise ValueError("沒有內容 parts")

            text = candidate.content.parts[0].text.strip()

            # ✅ JSON parsing
            result = json.loads(text)

            # ✅ 欄位驗證
            assert isinstance(result.get('sentiment_score'), int), \
                'sentiment_score 必須是整數'
            assert 1 <= result['sentiment_score'] <= 10, \
                'sentiment_score 必須在 1-10'

            # 有些模型會回 int → 強制轉 float
            confidence = float(result.get('confidence'))

            assert 0.0 <= confidence <= 1.0, \
                'confidence 必須在 0-1'

            result['confidence'] = confidence

            return result

        except Exception as e:
            wait = RETRY_BASE_DELAY * (2 ** (attempt - 1))
            log.warning(f'第 {attempt} 次嘗試失敗：{e}，等待 {wait}s 後重試...')
            time.sleep(wait)

    raise RuntimeError(f'已重試 {MAX_RETRIES} 次，仍然失敗')

def load_resume_line() -> int:
    """讀取斷點紀錄，回傳上次處理到的行號（0-indexed）"""
    if Path(RESUME_FILE).exists():
        try:
            return int(Path(RESUME_FILE).read_text().strip())
        except ValueError:
            pass
    return 0


def save_resume_line(line_num: int):
    """將當前進度寫入斷點檔"""
    Path(RESUME_FILE).write_text(str(line_num))


print('✅ 所有輔助函式定義完成')

c:\Users\user\miniconda3\envs\gemini-labeler\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


✅ 所有輔助函式定義完成


c:\Users\user\miniconda3\envs\gemini-labeler\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\user\AppData\Local\Temp\ipykernel_560\953405476.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## 🚀 Step 6 — 執行標注（主流程）

> 💡 **斷點續跑**：若中途中斷，直接重新執行此 Cell，
> 程式會自動從 `gemini_labeled_resume.txt` 中讀取上次進度，不重複計費。

In [34]:
log.info('=' * 60)
log.info('🚀 Gemini Labeler 啟動')
log.info(f'   模型  : {MODEL_NAME}')
log.info(f'   輸入  : {INPUT_FILE}')
log.info(f'   輸出  : {OUTPUT_FILE}')
log.info('=' * 60)

model       = init_model()
resume_from = load_resume_line()

if resume_from > 0:
    log.info(f'⏩ 偵測到斷點紀錄，從第 {resume_from + 1} 筆繼續')

# 計算總行數
total_lines = sum(1 for _ in open(INPUT_FILE, 'r', encoding='utf-8'))
log.info(f'   總筆數: {total_lines}')

success_count = 0
error_count   = 0
start_time    = datetime.now()

# 斷點續跑：有紀錄則 append，否則新建
write_mode = 'a' if resume_from > 0 else 'w'

with open(INPUT_FILE, 'r', encoding='utf-8') as f_in, \
     open(OUTPUT_FILE, write_mode, encoding='utf-8') as f_out:

    for line_num, line in enumerate(f_in):   # 0-indexed
        if line_num < resume_from:
            continue

        line = line.strip()
        if not line:
            continue

        display_num = line_num + 1

        try:
            data      = json.loads(line)
            news_text = data.get('input', '').strip()
            stock_id  = data.get('stock', '未知')

            if not news_text:
                log.warning(f'  第 {display_num} 筆：input 欄位為空，跳過')
                continue

            # 組合 Prompt
            prompt = (
                f'{SYSTEM_PROMPT}\n\n'
                f'分析目標股票代號：{stock_id}\n'
                f'新聞內容：{news_text}'
            )

            # 呼叫 API
            result = call_gemini_with_retry(model, prompt)

            # 寫入結果
            data['output'] = result
            f_out.write(json.dumps(data, ensure_ascii=False) + '\n')
            f_out.flush()

            save_resume_line(display_num)
            success_count += 1

            elapsed  = (datetime.now() - start_time).seconds
            progress = display_num / total_lines * 100
            log.info(
                f'✅ [{display_num:>5}/{total_lines}] {progress:5.1f}% | '
                f'股票:{stock_id:>6} | '
                f'分數:{result["sentiment_score"]:>2} | '
                f'信心:{result["confidence"]:.2f} | '
                f'已耗時:{elapsed}s'
            )

        except Exception as e:
            error_count += 1
            log.error(f'❌ 第 {display_num} 筆最終失敗：{e}')
            data['output'] = {
                'sentiment_score': None,
                'confidence': None,
                'error': str(e),
            }
            f_out.write(json.dumps(data, ensure_ascii=False) + '\n')
            f_out.flush()

        # 頻率控制
        time.sleep(SLEEP_BETWEEN_CALLS)

# 最終統計
total_elapsed = (datetime.now() - start_time).total_seconds()
log.info('=' * 60)
log.info('🎉 標注完成！')
log.info(f'   成功: {success_count} 筆')
log.info(f'   失敗: {error_count} 筆')
log.info(f'   總耗時: {total_elapsed:.0f}s（{total_elapsed/60:.1f} 分鐘）')
log.info(f'   輸出檔: {OUTPUT_FILE}')
log.info('=' * 60)

2026-06-29 16:57:37,281 [INFO] ============================================================
2026-06-29 16:57:37,282 [INFO] 🚀 Gemini Labeler 啟動
2026-06-29 16:57:37,282 [INFO]    模型  : gemini-3.1-flash-lite
2026-06-29 16:57:37,283 [INFO]    輸入  : data/jsonl/gemini_label/prelabeled.jsonl
2026-06-29 16:57:37,283 [INFO]    輸出  : data/jsonl/gemini_label/finish_labeled.jsonl
2026-06-29 16:57:37,284 [INFO] ============================================================
2026-06-29 16:57:37,284 [INFO] 模型初始化完成：gemini-3.1-flash-lite
2026-06-29 16:57:37,285 [INFO] ⏩ 偵測到斷點紀錄，從第 5394 筆繼續
2026-06-29 16:57:37,311 [INFO]    總筆數: 5356
2026-06-29 16:57:37,337 [INFO] ============================================================
2026-06-29 16:57:37,337 [INFO] 🎉 標注完成！
2026-06-29 16:57:37,338 [INFO]    成功: 0 筆
2026-06-29 16:57:37,338 [INFO]    失敗: 0 筆
2026-06-29 16:57:37,339 [INFO]    總耗時: 0s（0.0 分鐘）
2026-06-29 16:57:37,339 [INFO]    輸出檔: data/jsonl/gemini_label/finish_labeled.jsonl
2026-06-29 16:57:37,340 [INFO]

## 📊 Step 7 — 標注結果快速檢視

In [35]:
import json
from collections import Counter

scores = []
error_lines = 0
low_confidence_count = 0  # 🎯 新增：信心不足計數器
CONF_THRESHOLD = 0.7      # 設定門檻值

with open('data/jsonl/gemini_label/finish_labeled.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        data  = json.loads(line)
        label = data.get('output', {})
        score = label.get('sentiment_score')
        conf  = label.get('confidence')

        if score is not None:
            scores.append(score)
            # 🎯 檢查信心是否低於 0.5 (且確保 conf 不是 None)
            if conf is not None and conf < CONF_THRESHOLD:
                low_confidence_count += 1
        else:
            error_lines += 1

dist = Counter(scores)

print(f'📋 標注結果統計')
print(f'   有效筆數 : {len(scores)}')
print(f'   失敗筆數 : {error_lines}')
print(f'   低信心數 : {low_confidence_count} (Confidence < {CONF_THRESHOLD})') # 🎯 顯示結果
if scores:
    print(f'   平均分數 : {sum(scores)/len(scores):.2f}')
    print(f'   最低分數 : {min(scores)}')
    print(f'   最高分數 : {max(scores)}')
print()

print('   分數分佈：')
total = len(scores)
for s in range(1, 11):
    count = dist.get(s, 0)
    # 防止 dist 為空時的報錯
    max_val = max(dist.values()) if dist else 1
    bar   = '█' * int(count / max_val * 30)
    pct   = count / total * 100 if total else 0
    print(f'   {s:>2}分 | {bar:<30} {count:>4} ({pct:4.1f}%)')

📋 標注結果統計
   有效筆數 : 5026
   失敗筆數 : 1259
   低信心數 : 986 (Confidence < 0.7)
   平均分數 : 5.48
   最低分數 : 1
   最高分數 : 9

   分數分佈：
    1分 |                                  28 ( 0.6%)
    2分 | ███████                         261 ( 5.2%)
    3分 | ███████████                     423 ( 8.4%)
    4分 | ███████████████████████████     998 (19.9%)
    5分 | ██████████████████████████████ 1073 (21.3%)
    6分 | ██████████████                  535 (10.6%)
    7分 | ████████████████████            740 (14.7%)
    8分 | ████████████████████            747 (14.9%)
    9分 | ██████                          221 ( 4.4%)
   10分 |                                   0 ( 0.0%)


## 對信心過低的重新標註

In [36]:
import json
from pathlib import Path

# 設定你實際的 JSONL 檔案路徑
INPUT_FILE = 'data/jsonl/gemini_label/finish_labeled_fixed.jsonl'
CONFIDENCE_THRESHOLD = 0.7

def find_low_confidence_records(file_path, threshold=0.7):
    path = Path(file_path)
    if not path.exists():
        print(f"❌ 找不到檔案：{file_path}，請確認檔案路徑是否正確！")
        return

    print(f"🔍 開始掃描檔案: {file_path}")
    print(f"🛡️ 篩選標準: confidence < {threshold}")
    print("=" * 65)

    low_conf_count = 0
    total_count = 0

    with open(path, 'r', encoding='utf-8') as f:
        for idx, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            total_count += 1
            
            try:
                data = json.loads(line)
                # 兼容不同的欄位格式 (支援 output 或舊版的 gemini_label)
                label_data = data.get('output') or data.get('gemini_label') or {}
                
                score = label_data.get('sentiment_score')
                confidence = label_data.get('confidence')
                stock_id = data.get('stock', '未知')
                date = data.get('date', '未知')

                # 判定為需要重新驗證/低信心度的情況
                is_none = (score is None) or (confidence is None)
                is_low = (confidence is not None) and (confidence < threshold)

                if is_none:
                    print(f"⚠️  第 {idx:>5} 筆 | 股票: {stock_id:>5} | 日期: {date} | 🔴 情感分數或信心度為空 (Null)")
                    low_conf_count += 1
                elif is_low:
                    print(f"📌 第 {idx:>5} 筆 | 股票: {stock_id:>5} | 日期: {date} | 🟡 信心度偏低: {confidence:.2f} (分數: {score})")
                    low_conf_count += 1

            except json.JSONDecodeError:
                print(f"❌ 第 {idx:>5} 筆 | ⚠️ 該行 JSON 格式損壞無法解析")
                low_conf_count += 1

    print("=" * 65)
    print(f"📊 掃描統計結果：")
    print(f"   總檢查行數 : {total_count} 筆")
    print(f"   低信心/異常: {low_conf_count} 筆")
    if total_count > 0:
        print(f"   需要重跑率 : {low_conf_count / total_count * 100:.1f}%")

if __name__ == '__main__':
    find_low_confidence_records(INPUT_FILE, CONFIDENCE_THRESHOLD)

🔍 開始掃描檔案: data/jsonl/gemini_label/finish_labeled_fixed.jsonl
🛡️ 篩選標準: confidence < 0.7
📌 第    12 筆 | 股票:  2330 | 日期: 2025-01-02 | 🟡 信心度偏低: 0.40 (分數: 5)
📌 第    13 筆 | 股票:  2330 | 日期: 2025-01-02 | 🟡 信心度偏低: 0.40 (分數: 5)
📌 第    33 筆 | 股票:  2330 | 日期: 2025-01-03 | 🟡 信心度偏低: 0.40 (分數: 4)
📌 第    35 筆 | 股票:  2330 | 日期: 2025-01-03 | 🟡 信心度偏低: 0.40 (分數: 6)
📌 第    39 筆 | 股票:  2330 | 日期: 2025-01-03 | 🟡 信心度偏低: 0.40 (分數: 5)
📌 第    42 筆 | 股票:  2330 | 日期: 2025-01-04 | 🟡 信心度偏低: 0.40 (分數: 4)
📌 第    49 筆 | 股票:  2330 | 日期: 2025-01-05 | 🟡 信心度偏低: 0.40 (分數: 5)
📌 第    54 筆 | 股票:  2330 | 日期: 2025-01-06 | 🟡 信心度偏低: 0.40 (分數: 5)
📌 第   123 筆 | 股票:  2330 | 日期: 2025-01-07 | 🟡 信心度偏低: 0.40 (分數: 5)
📌 第   124 筆 | 股票:  2330 | 日期: 2025-01-07 | 🟡 信心度偏低: 0.40 (分數: 5)
📌 第   129 筆 | 股票:  2330 | 日期: 2025-01-07 | 🟡 信心度偏低: 0.40 (分數: 5)
📌 第   141 筆 | 股票:  2330 | 日期: 2025-01-08 | 🟡 信心度偏低: 0.40 (分數: 4)
📌 第   157 筆 | 股票:  2330 | 日期: 2025-01-08 | 🟡 信心度偏低: 0.30 (分數: 5)
📌 第   167 筆 | 股票:  2330 | 日期: 2025-01-09 | 🟡 信心度偏低: 0.40 (分數: 5)
📌 第

In [5]:
import json
import time
from pathlib import Path
from datetime import datetime
from IPython.display import display, HTML, clear_output

# ================== ⚙️ 重跑核心設定區 ==================
ORIGIN_FILE     = 'data/jsonl/gemini_label/qwen_predictions.jsonl'       # 原始已跑完包含低信心的檔案
NEW_OUTPUT_FILE = 'data/jsonl/gemini_label/qwen_predictions_fixed.jsonl' # 修正與整合後的最終完美成果檔
FIX_RESUME_FILE = 'gemini_fixed_resume.txt'                            # 💡 重跑任務專用的斷點紀錄檔
CONF_THRESHOLD  = 0.7                                                  # 篩選門檻

# 💡 既然都要重跑了，強烈建議將模型升級為智慧表現更穩健的級別（如 gemini-2.5-flash）
RETRY_MODEL_NAME = 'gemini-3.1-flash-lite' 
# ======================================================

def load_fix_resume_line() -> int:
    """讀取『重跑專用』的斷點紀錄，回傳上次處理到的行號（0-indexed）"""
    if Path(FIX_RESUME_FILE).exists():
        try:
            return int(Path(FIX_RESUME_FILE).read_text().strip())
        except ValueError:
            pass
    return 0

def save_fix_resume_line(line_num: int):
    """將當前重跑進度寫入重跑斷點檔"""
    Path(FIX_RESUME_FILE).write_text(str(line_num))


# --- 🤖 初始化重跑專用的模型環境 ---
old_model_name = MODEL_NAME
MODEL_NAME = RETRY_MODEL_NAME
retry_model = init_model()         # 自動調用你先前定義的 init_model() 函式
MODEL_NAME = old_model_name        # 還原全域變數設定

# --- 🔄 斷點狀態偵測 ---
resume_from = load_fix_resume_line()
write_mode = 'a' if resume_from > 0 else 'w'

# 計算來源檔總行數
total_rows = sum(1 for _ in open(ORIGIN_FILE, 'r', encoding='utf-8'))

# 歸零/初始化計數器
success_count = 0
skip_count    = 0
error_count   = 0
start_time    = datetime.now()
detailed_logs = []  # 用來收集需要重跑的明細，最後塞入摺疊框

print(f"🚀 標注修正制導流啟動...")
if resume_from > 0:
    print(f"⏩ 偵測到重跑斷點，將從第 {resume_from + 1} 筆記錄繼續（採用追加寫入模式）")
time.sleep(2)


# --- 🏃‍♂️ 開始主迴圈處理 ---
with open(ORIGIN_FILE, 'r', encoding='utf-8') as f_in, \
     open(NEW_OUTPUT_FILE, write_mode, encoding='utf-8') as f_out:
         
    for line_num, line in enumerate(f_in): # 0-indexed
        if line_num < resume_from:
            continue
            
        line = line.strip()
        if not line:
            continue
            
        display_num = line_num + 1
        data = json.loads(line)
        
        # 兼容不同時期的欄位結構
        label_data = data.get('output') or data.get('gemini_label') or {}
        confidence = label_data.get('confidence')
        score      = label_data.get('sentiment_score')
        stock_id   = data.get('stock', '未知')
        news_text  = data.get('input', '').strip()

        # 判定是否符合重跑資格 (Null 值或信心度低於門檻)
        is_none = (score is None) or (confidence is None)
        is_low  = (confidence is not None) and (confidence < CONF_THRESHOLD)

        if (is_none or is_low) and news_text:
            # 組合先前 Step 4 的 SYSTEM_PROMPT 基礎
            prompt = (
                f'{SYSTEM_PROMPT}\n\n'
                f'分析目標股票代號：{stock_id}\n'
                f'新聞內容：{news_text}'
            )
            
            try:
                # 調用你先前定義、帶指數退避機制的 API 呼叫函式
                result = call_gemini_with_retry(retry_model, prompt)
                
                # 更新覆蓋原本低信心的結果
                data['output'] = result
                success_count += 1
                detailed_logs.append(
                    f"♻️  修復成功 [{display_num:>5}/{total_rows}] | 股票: {stock_id:>5} | 新分數: {result['sentiment_score']} | 新信心: {result['confidence']:.2f}"
                )
            except Exception as e:
                error_count += 1
                detailed_logs.append(
                    f"❌ 修復失敗 [{display_num:>5}/{total_rows}] | 原因: {e} (已保留原資料防呆)"
                )
                
            # 嚴格遵循你的 RPM 間隔控制
            time.sleep(SLEEP_BETWEEN_CALLS)
        else:
            # 高信心資料直接跳過不重複呼叫，省錢！
            skip_count += 1

        # 寫入整合新檔案，並即時更新斷點
        f_out.write(json.dumps(data, ensure_ascii=False) + '\n')
        if display_num % 10 == 0:
            f_out.flush()
        save_fix_resume_line(display_num)

        # --- 📊 摺疊框核心前置：Jupyter 實時儀表板重新渲染 ---
        if display_num % 5 == 0 or display_num == total_rows:
            clear_output(wait=True)
            elapsed = (datetime.now() - start_time).seconds
            progress = (display_num / total_rows) * 100
            print("============================================================")
            print(" ⚙️  Gemini 標注器 — 低信心資料重跑監控儀表板")
            print("============================================================")
            print(f" 📈 當前進度 : {display_num:>5} / {total_rows} ({progress:.1f}%)")
            print(f" ⏭️  直接延用 : {skip_count:>5} 筆 (高信心封存)")
            print(f" ♻️  重跑成功 : {success_count:>5} 筆")
            print(f" ❌ 重跑失敗 : {error_count:>5} 筆")
            print(f" ⏱️  運作耗時 : {elapsed:>5} 秒")
            print("============================================================")
            if detailed_logs:
                print(f" 📝 執行動態: {detailed_logs[-1]}")


# --- 🎉 完美收尾：渲染 HTML 摺疊輸出框 ---
clear_output(wait=True)
total_elapsed = (datetime.now() - start_time).total_seconds()

print("============================================================")
print("🎉 恭喜！低信心資料重新標註與完整檔案整合任務全數完成！")
print("============================================================")
print(f" 總共檢查行數   : {total_rows} 筆")
print(f" 高信心延用行數 : {skip_count} 筆")
print(f" 成功修復重標數 : {success_count} 筆")
print(f" 依然無解錯誤數 : {error_count} 筆")
print(f" 總計執行時間   : {total_elapsed/60:.1f} 分鐘")
print(f" 最終成果輸出檔 : {NEW_OUTPUT_FILE}")
print("============================================================")

# 建立 HTML 的 <details> 摺疊標籤
html_log_content = "<br>".join(detailed_logs) if detailed_logs else "（本次無任何需要重跑的低信心資料）"
collapsible_html = f"""
<details style="border: 1px solid #1a73e8; border-radius: 6px; padding: 10px; margin-top: 15px; background-color: #f8f9fa;">
    <summary style="font-weight: bold; font-size: 14px; color: #1a73e8; cursor: pointer; user-select: none; padding: 5px;">
        🔍 點擊展開 / 摺疊詳細重跑修復歷史紀錄明細 (共處理 {len(detailed_logs)} 筆低信心變更)
    </summary>
    <div style="max-height: 250px; overflow-y: auto; padding: 12px; margin-top: 10px; font-family: 'Courier New', monospace; font-size: 12px; background-color: #ffffff; border: 1px solid #dadce0; border-radius: 4px; line-height: 1.6; color: #3c4043;">
        {html_log_content}
    </div>
</details>
"""
# 呼叫 Jupyter 的 display 顯示精美摺疊區
display(HTML(collapsible_html))

🎉 恭喜！低信心資料重新標註與完整檔案整合任務全數完成！
 總共檢查行數   : 1904 筆
 高信心延用行數 : 1904 筆
 成功修復重標數 : 0 筆
 依然無解錯誤數 : 0 筆
 總計執行時間   : 0.1 分鐘
 最終成果輸出檔 : data/jsonl/gemini_label/qwen_predictions_fixed.jsonl


## 比對資料數量

In [41]:
import json
from pathlib import Path

# ================== 🎯 依你指定設定檔案路徑 ==================
PRELABELED_MASTER = 'data/jsonl/gemini_label/prelabeled copy.jsonl' # 原始真理母檔
FIXED_LABEL_POOL  = 'data/jsonl/gemini_label/finish_labeled_fixed.jsonl' # 含有重複與低信心的成果檔

# 輸出的全新對齊檔案
CLEAN_PRELABELED  = 'data/jsonl/gemini_label/prelabeled_fixed.jsonl'
CLEAN_FINAL_LABEL = 'data/jsonl/gemini_label/finish_labeled_fixed_2.jsonl'
# ============================================================

print("🛠️  開始執行『指定雙階段欄位對比剪裁機制』...")

# ------------------------------------------------------------
# 【第一階段】比對固定欄位，將 fixed.jsonl 內重複出現的筆數刪除
# ------------------------------------------------------------
fixed_seen = set()
unique_fixed_data = []
fixed_duplicate_count = 0
total_fixed_rows = 0

with open(FIXED_LABEL_POOL, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        total_fixed_rows += 1
        data = json.loads(line)
        
        # 擷取你指定的關鍵比對欄位
        date_val  = str(data.get('date', '')).strip()
        stock_val = str(data.get('stock', '')).strip()
        input_val = str(data.get('input', '')).strip()
        
        # 建立黃金複合 Key
        composite_key = (date_val, stock_val, input_val)
        
        if composite_key in fixed_seen:
            fixed_duplicate_count += 1
        else:
            fixed_seen.add(composite_key)
            unique_fixed_data.append(data)

print(f"   ✅ 第一階段完成：")
print(f"      - 標注檔掃描總數 : {total_fixed_rows} 筆")
print(f"      - 抓到並刪除重複 : {fixed_duplicate_count} 筆")
print(f"      - 處理後剩餘獨一 : {len(unique_fixed_data)} 筆")


# ------------------------------------------------------------
# 【第二階段】將 prelabeled 內不存在於處理後 fixed 檔案中的資料刪除
# ------------------------------------------------------------
clean_prelabeled_data = []
prelabeled_deleted_count = 0
total_prelabeled_rows = 0

with open(PRELABELED_MASTER, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        total_prelabeled_rows += 1
        data = json.loads(line)
        
        # 用同樣的標準擷取母檔欄位
        date_val  = str(data.get('date', '')).strip()
        stock_val = str(data.get('stock', '')).strip()
        input_val = str(data.get('input', '')).strip()
        
        composite_key = (date_val, stock_val, input_val)
        
        # 核心對比：檢查這筆母檔資料，是否存在於剛剛處理乾淨的 fixed 集合中
        if composite_key in fixed_seen:
            clean_prelabeled_data.append(data)
        else:
            prelabeled_deleted_count += 1

print(f"   ✅ 第二階段完成：")
print(f"      - 母檔掃描總數   : {total_prelabeled_rows} 筆")
print(f"      - 刪除不存在資料 : {prelabeled_deleted_count} 筆")
print(f"      - 同步後保留筆數 : {len(clean_prelabeled_data)} 筆")


# ------------------------------------------------------------
# 【第三階段】將最終完美成果寫入對應的輸出路徑
# ------------------------------------------------------------
with open(CLEAN_FINAL_LABEL, 'w', encoding='utf-8') as f_out:
    for row in unique_fixed_data:
        f_out.write(json.dumps(row, ensure_ascii=False) + '\n')

with open(CLEAN_PRELABELED, 'w', encoding='utf-8') as f_out:
    for row in clean_prelabeled_data:
        f_out.write(json.dumps(row, ensure_ascii=False) + '\n')


# ================== 📊 終極精煉大驗收 ==================
final_in_count  = sum(1 for _ in open(CLEAN_PRELABELED, 'r', encoding='utf-8'))
final_out_count = sum(1 for _ in open(CLEAN_FINAL_LABEL, 'r', encoding='utf-8'))

print("=" * 60)
print("🎉 依指定邏輯剪裁對齊任務全數完成！")
print("============================================================")
print(f" 📂 最終預預測檔 總行數 ({CLEAN_PRELABELED}) : {final_in_count:>5} 筆")
print(f" 🎯 最終標注成果 總行數 ({CLEAN_FINAL_LABEL}) : {final_out_count:>5} 筆")
print("============================================================")

if final_in_count == final_out_count:
    print("✨ 太完美了！兩邊數值達成 100% 絕對對齊！")
    print("   所有的重複垃圾、歷史舊新聞、頑固低信心已經全部被精準剔除。")
else:
    print("⚠️ 警告：兩邊數量仍有微小落差，可能母檔本身含有重複項，建議檢查資料庫。")
print("============================================================")

🛠️  開始執行『指定雙階段欄位對比剪裁機制』...
   ✅ 第一階段完成：
      - 標注檔掃描總數 : 5775 筆
      - 抓到並刪除重複 : 0 筆
      - 處理後剩餘獨一 : 5775 筆
   ✅ 第二階段完成：
      - 母檔掃描總數   : 5816 筆
      - 刪除不存在資料 : 41 筆
      - 同步後保留筆數 : 5775 筆
🎉 依指定邏輯剪裁對齊任務全數完成！
 📂 最終預預測檔 總行數 (data/jsonl/gemini_label/prelabeled_fixed.jsonl) :  5775 筆
 🎯 最終標注成果 總行數 (data/jsonl/gemini_label/finish_labeled_fixed_2.jsonl) :  5775 筆
✨ 太完美了！兩邊數值達成 100% 絕對對齊！
   所有的重複垃圾、歷史舊新聞、頑固低信心已經全部被精準剔除。


## 刪除多餘的資料

In [ ]:
import json
import os
from pathlib import Path
from IPython.display import display, HTML

# ================== 🧼 資料清洗路徑設定 ==================
INPUT_FILE  = 'data/jsonl/taiwan_news/2454.jsonl'         # 原始預預測檔
OUTPUT_FILE = 'data/jsonl/finish_labeled/2454_20250101-20260630.jsonl' # 帶有 11.9% 低信心的標注檔
FINAL_FILE  = 'data/jsonl/gemini_label/finish_labeled.jsonl' # 清洗後的最終完美標注檔

CONFIDENCE_THRESHOLD = 0.7
# ======================================================

print("🧼 頑固低信心資料深度清洗精煉流，啟動...")
print(f"   Step 1. 正在掃描標注檔: {OUTPUT_FILE}")

valid_signatures = set()  # 用來儲存所有高信心資料的特徵組
dropped_logs = []         # 收集被刪除的紀錄明細
total_output_rows = 0
keep_count = 0

# --- 階段 1：掃描標注檔，篩選並產出最終高信心檔案 ---
with open(OUTPUT_FILE, 'r', encoding='utf-8') as f_in, \
     open(FINAL_FILE, 'w', encoding='utf-8') as f_out:
    
    for idx, line in enumerate(f_in, start=1):
        line = line.strip()
        if not line:
            continue
        total_output_rows += 1
        
        try:
            data = json.loads(line)
            label_data = data.get('output') or data.get('gemini_label') or {}
            score = label_data.get('sentiment_score')
            confidence = label_data.get('confidence')
            stock_id = data.get('stock', '未知')
            date = data.get('date', '未知')
            news_text = data.get('input', '').strip()
            
            is_none = (score is None) or (confidence is None)
            is_low = (confidence is not None) and (confidence < CONFIDENCE_THRESHOLD)
            
            # 建立該筆資料的唯一特徵組（股票 + 新聞內容前 20 字），防止行號錯位
            signature = (stock_id, news_text[:20])
            
            if is_none or is_low:
                # 抓出低信心，丟進刪除日誌，不寫入新檔案
                dropped_logs.append(f"🗑️  已剔除第 {idx:>5} 筆 | 股票: {stock_id:>5} | 日期: {date} | 分數: {score} | 信心度: {confidence}")
            else:
                # 高信心資料：記錄特徵，寫入 FINAL_FILE
                valid_signatures.add(signature)
                f_out.write(json.dumps(data, ensure_ascii=False) + '\n')
                keep_count += 1
                
        except json.JSONDecodeError:
            dropped_logs.append(f"❌ 已剔除第 {idx:>5} 筆 | 原因: JSON 格式損壞無法解析")

print(f"   ↳ 標注檔過濾完畢。保留: {keep_count} 筆，剔除: {len(dropped_logs)} 筆。")
print(f"   Step 2. 正在同步清洗原始預預測檔: {INPUT_FILE}")


# --- 階段 2：同步清洗原始 prelabeled.jsonl 檔案 ---
TEMP_INPUT_FILE = INPUT_FILE + '.tmp'
total_input_rows = 0
input_keep_count = 0

with open(INPUT_FILE, 'r', encoding='utf-8') as f_in, \
     open(TEMP_INPUT_FILE, 'w', encoding='utf-8') as f_out:
         
    for line in f_in:
        line = line.strip()
        if not line:
            continue
        total_input_rows += 1
        
        try:
            data = json.loads(line)
            stock_id = data.get('stock', '未知')
            news_text = data.get('input', '').strip()
            signature = (stock_id, news_text[:20])
            
            # 只有在剛才標注檔被判定為「高信心」的特徵，才允許寫入
            if signature in valid_signatures:
                f_out.write(json.dumps(data, ensure_ascii=False) + '\n')
                input_keep_count += 1
        except:
            pass # 損壞欄位直接不保留

# 安全地用清洗後的暫存檔覆蓋掉原始預預測檔
if os.path.exists(TEMP_INPUT_FILE):
    os.replace(TEMP_INPUT_FILE, INPUT_FILE)

print("   ↳ 預預測檔過濾完畢。")


# --- 📊 終極精煉成果報告 ---
print("=" * 60)
print("🎉 雙向資料集精煉清洗任務順利完成！")
print("============================================================")
print(f" 📂 標注成果檔 ({OUTPUT_FILE}) -> ({FINAL_FILE})")
print(f"    - 原始總筆數 : {total_output_rows} 筆")
print(f"    - 清除黃金高信心 : {keep_count} 筆")
print(f"    - 徹底拔除雜訊數 : {len(dropped_logs)} 筆")
print("-" * 60)
print(f" 📂 原始預預測檔 ({INPUT_FILE})")
print(f"    - 同步清洗前總數 : {total_input_rows} 筆")
print(f"    - 同步清洗後剩餘 : {input_keep_count} 筆")
print("============================================================")
print("============================================================")

# 渲染精美 HTML 摺疊明細框
html_log_content = "<br>".join(dropped_logs) if dropped_logs else "（無任何剔除資料）"
collapsible_html = f"""
<details style="border: 1px solid #ea4335; border-radius: 6px; padding: 10px; margin-top: 15px; background-color: #fce8e6;">
    <summary style="font-weight: bold; font-size: 14px; color: #ea4335; cursor: pointer; user-select: none; padding: 5px;">
        🔍 點擊展開 / 摺疊本次被永久抹除的 {len(dropped_logs)} 筆低信心資料明細清單
    </summary>
    <div style="max-height: 250px; overflow-y: auto; padding: 12px; margin-top: 10px; font-family: 'Courier New', monospace; font-size: 12px; background-color: #ffffff; border: 1px solid #fad2cf; border-radius: 4px; line-height: 1.6; color: #202124;">
        {html_log_content}
    </div>
</details>
"""
display(HTML(collapsible_html))

🧼 頑固低信心資料深度清洗精煉流，啟動...
   Step 1. 正在掃描標注檔: data/jsonl/finish_labeled/2454_20250101-20260630.jsonl
   ↳ 標注檔過濾完畢。保留: 2837 筆，剔除: 1 筆。
   Step 2. 正在同步清洗原始預預測檔: data/jsonl/taiwan_news/2454.jsonl
   ↳ 預預測檔過濾完畢。
🎉 雙向資料集精煉清洗任務順利完成！
 📂 標注成果檔 (data/jsonl/finish_labeled/2454_20250101-20260630.jsonl) -> (data/jsonl/gemini_label/finish_labeled.jsonl)
    - 原始總筆數 : 2838 筆
    - 清除黃金高信心 : 2837 筆
    - 徹底拔除雜訊數 : 1 筆
------------------------------------------------------------
 📂 原始預預測檔 (data/jsonl/taiwan_news/2454.jsonl)
    - 同步清洗前總數 : 2838 筆
    - 同步清洗後剩餘 : 2837 筆
💡 提示：現在這兩個檔案在數量與順序上已完美 100% 對齊，可以直接進入 Qwen 預測步驟！


## ✅ 完成！下一步

標注完成後，請開啟 **`qwen_predictor_colab.ipynb`**，
將 `gemini_labeled.jsonl` 上傳至 Google Colab，
使用 Qwen2.5-14B-Instruct-AWQ 進行批次推論預測。

```
gemini_labeled.jsonl  ← 本 Notebook 的輸出
        ↓
[上傳至 Google Colab]
        ↓
qwen_predictor_colab.ipynb
```